# Assinantes pagantes ativos no corte (30/09/2026)

**Pergunta:** quantos assinantes pagantes ativos a Vitaliza tinha no corte, 30/09/2026?

**Fontes:** `dados/brutos/assinaturas.csv` e `dados/brutos/usuarios.csv` (dicionário em `docs/`).

**Regras:**
- contas de teste (e-mail `@vitaliza.com.br`) não contam;
- as linhas são mostradas depois de cada filtro;
- nada em `dados/brutos/` é alterado (o hash dos arquivos é conferido no início e no fim).

**Definição usada:** assinatura com `status_atual == "ativa"` no retrato de 01/10/2026, iniciada até o corte,
de conta que não é de teste, em plano pago (mensal ou anual). Assinaturas que expiraram em 30/09/2026 **não** contam.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
SAIDAS = ROOT / "saidas"
CORTE = pd.Timestamp("2026-09-30")

ARQUIVOS = ["assinaturas.csv", "usuarios.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
hash_inicio

{'assinaturas.csv': 'd1e15bd80a4d48750795387d1f3dc98cd9c2bcdf',
 'usuarios.csv': '00422a4eb2763dbe37db85d97729228f33185409'}

## 1. Leitura

Tudo como texto, para nenhuma conversão automática esconder problema de formato.

In [2]:
assinaturas = pd.read_csv(BRUTOS / "assinaturas.csv", dtype=str)
usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str)

# Linhas esperadas segundo o dicionário de dados
assert len(assinaturas) == 32_643
assert len(usuarios) == 64_948
print(f"assinaturas: {assinaturas.shape[0]:,} linhas x {assinaturas.shape[1]} colunas")
print(f"usuarios:    {usuarios.shape[0]:,} linhas x {usuarios.shape[1]} colunas")
assinaturas.head()

assinaturas: 32,643 linhas x 9 colunas
usuarios:    64,948 linhas x 12 colunas


,assinatura_id,user_id,plano_atual,canal_cobranca,inicio,status_atual,renovacao_automatica,data_fim,motivo_fim
0,s_000000,u_f4d898b7,anual,app_store,2023-10-25,ativa,nao,NaN,NaN
1,s_000001,u_066eb85c,mensal,web_cartao,2021-11-01,expirada,nao,2025-08-12,renovacao_desligada
2,s_000002,u_fec25b29,mensal,google_play,2020-10-29,expirada,nao,2024-09-08,renovacao_desligada
3,s_000003,u_b532a8ab,anual,app_store,2024-01-19,expirada,nao,2025-01-18,renovacao_desligada
4,s_000004,u_666cc36a,anual,google_play,2023-05-05,expirada,nao,2026-05-04,renovacao_desligada


## 2. Qualidade

Identificadores, ligação entre os arquivos, datas e categorias.

In [3]:
checagens = {
    "assinatura_id duplicado": assinaturas["assinatura_id"].duplicated().sum(),
    "linhas inteiras duplicadas em assinaturas": assinaturas.duplicated().sum(),
    "user_id duplicado em usuarios": usuarios["user_id"].duplicated().sum(),
    "user_id de assinatura sem cadastro": (~assinaturas["user_id"].isin(usuarios["user_id"])).sum(),
    "e-mail vazio em usuarios": usuarios["email"].isna().sum(),
}
pd.Series(checagens, name="linhas")

assinatura_id duplicado                      0
linhas inteiras duplicadas em assinaturas    0
user_id duplicado em usuarios                0
user_id de assinatura sem cadastro           0
e-mail vazio em usuarios                     0
Name: linhas, dtype: int64

In [4]:
# Datas: todas no formato AAAA-MM-DD? Alguma depois do corte?
for col in ["inicio", "data_fim"]:
    valores = assinaturas[col].dropna()
    datas = pd.to_datetime(valores, format="%Y-%m-%d", errors="coerce")
    print(f"{col}: {len(valores):,} preenchidas | fora do formato: {datas.isna().sum()} | "
          f"min {datas.min().date()} | max {datas.max().date()} | depois do corte: {(datas > CORTE).sum()}")

assinaturas["inicio"] = pd.to_datetime(assinaturas["inicio"], format="%Y-%m-%d")
assinaturas["data_fim"] = pd.to_datetime(assinaturas["data_fim"], format="%Y-%m-%d")

inicio: 32,643 preenchidas | fora do formato: 0 | min 2020-09-08 | max 2026-09-30 | depois do corte: 0
data_fim: 17,830 preenchidas | fora do formato: 0 | min 2024-07-07 | max 2026-09-30 | depois do corte: 0


O arquivo é um retrato de 01/10/2026, mas nenhuma assinatura começa ou termina depois de 30/09/2026.
Então o `status_atual` do retrato descreve a base no fim do dia 30/09, e não é preciso reconstruir a
situação pelos eventos.

In [5]:
# Categorias: a padronização (minúsculas, sem espaços nas pontas) muda alguma coisa?
CATEGORICAS = ["plano_atual", "canal_cobranca", "status_atual", "renovacao_automatica", "motivo_fim"]
for col in CATEGORICAS:
    padronizado = assinaturas[col].str.strip().str.lower()
    mudou = (padronizado.fillna("") != assinaturas[col].fillna("")).sum()
    assinaturas[col] = padronizado
    print(f"{col}: {mudou} valores alterados pela padronização -> {assinaturas[col].value_counts(dropna=False).to_dict()}")

usuarios["email"] = usuarios["email"].str.strip().str.lower()

plano_atual: 0 valores alterados pela padronização -> {'mensal': 19556, 'anual': 13027, 'cortesia': 60}
canal_cobranca: 0 valores alterados pela padronização -> {'app_store': 11170, 'google_play': 10948, 'web_cartao': 7087, 'web_pix': 3378, 'interno': 60}
status_atual: 0 valores alterados pela padronização -> {'expirada': 17758, 'ativa': 14813, 'reembolsada': 72}
renovacao_automatica: 0 valores alterados pela padronização -> {'nao': 20140, 'sim': 12503}
motivo_fim: 0 valores alterados pela padronização -> {'renovacao_desligada': 16145, nan: 14813, 'falha_pagamento': 1321, 'chargeback': 292, 'reembolso': 72}


In [6]:
# Coerência: assinatura ativa não tem data de fim; encerrada tem
pd.crosstab(assinaturas["status_atual"], assinaturas["data_fim"].notna().rename("tem data_fim"))

tem data_fim,False,True
status_atual,,
ativa,14813,0
expirada,0,17758
reembolsada,0,72


O dicionário lista só `mensal`/`anual` e quatro canais de cobrança. O arquivo traz também o plano
`cortesia` com canal `interno`, que o dicionário não descreve. O cruzamento abaixo mostra que são
exatamente as contas de teste.

In [7]:
eh_teste = usuarios.set_index("user_id")["email"].str.endswith("@vitaliza.com.br")
pd.crosstab(
    [assinaturas["plano_atual"], assinaturas["canal_cobranca"]],
    assinaturas["user_id"].map(eh_teste).rename("conta de teste"),
)

conta de teste              False  True 
plano_atual canal_cobranca              
anual       app_store        4552      0
            google_play      4529      0
            web_cartao       2859      0
            web_pix          1087      0
cortesia    interno             0     60
mensal      app_store        6618      0
            google_play      6419      0
            web_cartao       4228      0
            web_pix          2291      0

## 3. Filtros

Cada filtro registra quantas linhas sobraram e quantas saíram.

In [8]:
registro = []

def registrar(etapa, criterio, df):
    anterior = registro[-1]["linhas"] if registro else len(df)
    registro.append({"etapa": etapa, "criterio": criterio, "linhas": len(df), "removidas": anterior - len(df)})
    print(f"{etapa}: {len(df):,} linhas ({anterior - len(df):,} removidas)")
    return df

base = registrar("0. Todas as assinaturas", "assinaturas.csv sem filtro", assinaturas)

base = registrar("1. Iniciadas até o corte", "inicio <= 2026-09-30",
                 base[base["inicio"] <= CORTE])

expiradas_no_corte = (base["data_fim"] == CORTE).sum()
print(f"   (dessas, {expiradas_no_corte} expiraram em 30/09/2026 e saem no próximo filtro)")

base = registrar("2. Ativas no corte", "status_atual == 'ativa' (sem data_fim)",
                 base[base["status_atual"] == "ativa"])
assert base["data_fim"].isna().all()

n_antes = len(base)
base = base.merge(usuarios[["user_id", "email"]], on="user_id", how="left", validate="many_to_one")
assert len(base) == n_antes and base["email"].notna().all()
base = registrar("3. Junção com usuarios", "left join por user_id (confere que não ganha nem perde linha)", base)

base = registrar("4. Sem contas de teste", "e-mail não termina em @vitaliza.com.br",
                 base[~base["email"].str.endswith("@vitaliza.com.br")])

base = registrar("5. Um assinante por conta", "user_id único",
                 base.drop_duplicates("user_id"))

# O que sobrou é tudo pagante: plano e canal do dicionário
assert set(base["plano_atual"]) == {"mensal", "anual"}
assert set(base["canal_cobranca"]) == {"app_store", "google_play", "web_cartao", "web_pix"}

filtros = pd.DataFrame(registro)
filtros

0. Todas as assinaturas: 32,643 linhas (0 removidas)
1. Iniciadas até o corte: 32,643 linhas (0 removidas)
   (dessas, 29 expiraram em 30/09/2026 e saem no próximo filtro)
2. Ativas no corte: 14,813 linhas (17,830 removidas)


3. Junção com usuarios: 14,813 linhas (0 removidas)
4. Sem contas de teste: 14,753 linhas (60 removidas)
5. Um assinante por conta: 14,753 linhas (0 removidas)


,etapa,criterio,linhas,removidas
0,0. Todas as assinaturas,assinaturas.csv sem filtro,32643,0
1,1. Iniciadas até o corte,inicio <= 2026-09-30,32643,0
2,2. Ativas no corte,status_atual == 'ativa' (sem data_fim),14813,17830
3,3. Junção com usuarios,left join por user_id (confere que não ganha n...,14813,0
4,4. Sem contas de teste,e-mail não termina em @vitaliza.com.br,14753,60
5,5. Um assinante por conta,user_id único,14753,0


In [9]:
TOTAL = len(base)
print(f"Assinantes pagantes ativos em 30/09/2026: {TOTAL:,}".replace(",", "."))

Assinantes pagantes ativos em 30/09/2026: 14.753


## 4. Conferência: plano e canal somam o total

In [10]:
por_plano = base["plano_atual"].value_counts().rename("assinantes")
por_canal = base["canal_cobranca"].value_counts().rename("assinantes")
plano_canal = pd.crosstab(base["plano_atual"], base["canal_cobranca"], margins=True, margins_name="total")

assert por_plano.sum() == TOTAL
assert por_canal.sum() == TOTAL
assert plano_canal.loc["total", "total"] == TOTAL
assert (plano_canal.drop(index="total")["total"] == por_plano.reindex(plano_canal.index.drop("total"))).all()
assert (plano_canal.drop(columns="total").loc["total"] == por_canal.reindex(plano_canal.columns.drop("total"))).all()
print(f"plano: {por_plano.sum():,} | canal: {por_canal.sum():,} | total: {TOTAL:,} -> batem")

display(por_plano.to_frame())
display(por_canal.to_frame())
plano_canal

plano: 14,753 | canal: 14,753 | total: 14,753 -> batem


,assinantes
plano_atual,
anual,8478
mensal,6275


,assinantes
canal_cobranca,
app_store,5395
google_play,4964
web_cartao,2994
web_pix,1400


canal_cobranca,app_store,google_play,web_cartao,web_pix,total
plano_atual,,,,,
anual,3047,2928,1803,700,8478
mensal,2348,2036,1191,700,6275
total,5395,4964,2994,1400,14753


## 5. Saídas e integridade dos brutos

In [11]:
filtros.to_csv(SAIDAS / "assinantes_ativos_corte_filtros.csv", index=False)
plano_canal.to_csv(SAIDAS / "assinantes_ativos_corte_plano_canal.csv")

hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/brutos/ inalterado: True
